In [ ]:
# ============================================================
#  SCVELO BENCHMARK — PURE SCVELO PIPELINE
# ============================================================

import numpy as np
import anndata
import scvelo as scv
import matplotlib.pyplot as plt
import scanpy as sc

# -----------------------------
# GLOBAL STORAGE DICTIONARY
# -----------------------------
try:
    scvelo_results
except NameError:
    scvelo_results = {}

# -----------------------------
# DATASET REGISTRY
# -----------------------------
DATASETS = {
    "cell_cycle": {
        "X": "./data/real_data_benchmark/cell_cycle/X_cc.npy",
        "V": "./data/real_data_benchmark/cell_cycle/V_cc.npy",
        "color": "./data/real_data_benchmark/cell_cycle/color_cell_cycle_relativePos.npy",
    },
    "pancreas": {
        "X": "./data/real_data_benchmark/pancreas/X_pca.npy",
        "V": "./data/real_data_benchmark/pancreas/V_pca_stochastic.npy",
        "color": "./data/real_data_benchmark/pancreas/pseudotime.npy",
    },
    "dentate_gyrus": {
        "X": "./data/real_data_benchmark/dentate_gyrus/X_pca.npy",
        "V": "./data/real_data_benchmark/dentate_gyrus/V_pca_dynamical.npy",
        "color": "./data/real_data_benchmark/dentate_gyrus/pseudotime.npy",
    },
    "larry": {
        "X": "./data/real_data_benchmark/larry/X_raw.npy",
        "V": "./data/real_data_benchmark/larry/V_raw.npy",
        "color": "./data/real_data_benchmark/larry/distance_pseudotime.npy",
    },
}

# -----------------------------
# SELECT DATASET
# -----------------------------
dataset_name = "cell_cycle"
cfg = DATASETS[dataset_name]

# -----------------------------
# LOAD DATA
# -----------------------------
X = np.load(cfg["X"])
V = np.load(cfg["V"])
color = np.load(cfg["color"])

# -----------------------------
# BUILD ANNADATA
# -----------------------------
adata = anndata.AnnData(X)

adata.layers["position"] = X
adata.layers["velocity"] = V

adata.obs["color"] = np.asarray(color, dtype=float)

# -----------------------------
# NEIGHBOR GRAPH
# -----------------------------
scv.pp.neighbors(
    adata,
    n_neighbors=30,
    use_rep="X"
)

# -----------------------------
# COMPUTE SCVELO UMAP
# -----------------------------
sc.tl.umap(
    adata,
    min_dist=0.6
)

# -----------------------------
# VELOCITY GRAPH
# -----------------------------
scv.tl.velocity_graph(
    adata,
    xkey="position",
    vkey="velocity"
)

# -----------------------------
# PROJECT VELOCITIES
# -----------------------------
scv.tl.velocity_embedding(
    adata,
    basis="umap"
)

# -----------------------------
# EPSILON JITTER
# -----------------------------
np.random.seed(0)

eps = 1e-6

adata.obsm["X_umap"] += eps * np.random.randn(*adata.obsm["X_umap"].shape)

adata.obsm["velocity_umap"] += eps * np.random.randn(
    *adata.obsm["velocity_umap"].shape
)

# -----------------------------
# STORE RESULTS
# -----------------------------
scvelo_results[dataset_name] = adata

# -----------------------------
# PLOT
# -----------------------------
fig, ax = plt.subplots(figsize=(6, 5))

scv.pl.velocity_embedding_stream(
    adata,
    basis="umap",
    color="color",
    color_map="viridis",
    arrow_size=3.0,
    linewidth=2.8,
    density=0.5,
    size=660,
    alpha=0.15,
    legend_loc=None,
    colorbar=False,
    show=False,
    ax=ax,
)

ax.set_axis_off()

plt.tight_layout()
save_path = f"./figures/benchmark/{dataset_name}_scVelo_streamline.png"

plt.savefig(
    save_path,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0,
    transparent=True,
)

print(f"Saved to: {save_path}")

plt.show()

In [ ]:
# ============================================================
#  LARRY DATA — FULL SCVELO PROCEDURE BENCHMARK
# ============================================================

import numpy as np
import anndata
import scanpy as sc
import scvelo as scv
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA

# -----------------------------
# GLOBAL STORAGE DICT
# -----------------------------
try:
    scvelo_results
except NameError:
    scvelo_results = {}

# -----------------------------
# SELECT DATASET
# -----------------------------
dataset_name = "larry"

cfg = DATASETS[dataset_name]

# -----------------------------
# LOAD RAW DATA
# -----------------------------
X = np.load(cfg["X"])
V = np.load(cfg["V"])
color = np.load(cfg["color"])

print(f"Loaded shapes: X={X.shape}, V={V.shape}")

# -----------------------------
# BUILD ANNADATA
# -----------------------------
adata = anndata.AnnData(X)

adata.layers["position"] = X
adata.layers["velocity"] = V

# -----------------------------
# COLOR NORMALIZATION
# -----------------------------
color = np.asarray(color, dtype=float)
color = np.nan_to_num(color, nan=0.0)

cmin, cmax = color.min(), color.max()

if cmax > cmin:
    color = (color - cmin) / (cmax - cmin)
else:
    color = np.zeros_like(color)

adata.obs["color"] = color

# -----------------------------
# PCA
# -----------------------------
n_pcs = 30

print(f"Running PCA ({n_pcs} PCs)...")

pca = PCA(
    n_components=n_pcs,
    random_state=0
)

X_pca = pca.fit_transform(X)

adata.obsm["X_pca"] = X_pca

# velocity projection into PCA
components = pca.components_.T
V_pca = V @ components

adata.obsm["velocity_pca"] = V_pca

print("X_pca:", X_pca.shape)
print("V_pca:", V_pca.shape)

# -----------------------------
# NEIGHBOR GRAPH
# -----------------------------
scv.pp.neighbors(
    adata,
    n_neighbors=30,
    use_rep="X_pca"
)

# -----------------------------
# UMAP
# -----------------------------
sc.tl.umap(
    adata,
    min_dist=0.5
)

# -----------------------------
# VELOCITY GRAPH
# -----------------------------
scv.tl.velocity_graph(
    adata,
    xkey="position",
    vkey="velocity"
)

# -----------------------------
# VELOCITY EMBEDDING
# -----------------------------
scv.tl.velocity_embedding(
    adata,
    basis="umap",
    vkey="velocity"
)

# -----------------------------
# EPSILON JITTER
# -----------------------------
np.random.seed(0)

eps = 1e-6

adata.obsm["X_umap"] += eps * np.random.randn(
    *adata.obsm["X_umap"].shape
)

adata.obsm["velocity_umap"] += eps * np.random.randn(
    *adata.obsm["velocity_umap"].shape
)

# -----------------------------
# SAVE
# -----------------------------
scvelo_results[dataset_name] = adata

print(f"\nSaved to scvelo_results['{dataset_name}']")

# -----------------------------
# PLOT
# -----------------------------
fig, ax = plt.subplots(figsize=(6, 5))

scv.pl.velocity_embedding_stream(
    adata,
    basis="umap",
    color="color",
    color_map="viridis",
    density=0.5,
    linewidth=2.5,
    arrow_size=1.5,
    size=660,
    alpha=0.15,
    legend_loc=None,
    colorbar=False,
    show=False,
    ax=ax,
)

ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------
# COLOR: CLIP UPPER 95% QUANTILE
# -----------------------------
color = np.asarray(color, dtype=float)

color = np.nan_to_num(
    color,
    nan=0.0,
    posinf=0.0,
    neginf=0.0,
)

q95 = np.quantile(color, 0.95)

color = np.clip(color, None, q95)

adata.obs["color"] = color

# -----------------------------
# COMPUTE ROBUST AXIS LIMITS
# -----------------------------
emb = adata.obsm["X_umap"]

x = emb[:, 0]
y = emb[:, 1]

xpad = 0.03 * (np.percentile(x, 99) - np.percentile(x, 1))
ypad = 0.03 * (np.percentile(y, 99) - np.percentile(y, 1))

xmin = np.percentile(x, 1) - xpad
xmax = np.percentile(x, 99) + xpad

ymin = np.percentile(y, 1) - ypad
ymax = np.percentile(y, 99) + ypad

# -----------------------------
# PLOT
# -----------------------------
fig, ax = plt.subplots(figsize=(6, 5))

scv.pl.velocity_embedding_stream(
    adata,
    basis="umap",
    color="color",
    color_map="viridis",
    colorbar=False,
    legend_loc=None,
    ax=ax,
    show=False,

    # points
    size=100,
    alpha=0.06,

    # streamlines
    density=1.,
    linewidth=2.2,
    arrow_size=1.3,

    # cleaner rendering
    smooth=0.8,
)

# -----------------------------
# CLEAN AXES
# -----------------------------
# ax.set_xlim(xmin, xmax)
# ax.set_ylim(ymin, ymax)

ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_title("")

plt.tight_layout(pad=0.1)

save_path = f"./figures/benchmark/{dataset_name}_scVelo_streamline.png"

plt.savefig(
    save_path,
    dpi=600,
    bbox_inches="tight",
    pad_inches=0,
    transparent=True,
)

print(f"Saved to: {save_path}")

plt.show()